# power_alloc.py —— 函数级 API 参考与算法走读

本 notebook 逐函数走读 [power_alloc.py](file:///./power_alloc.py) 的实现细节，聚焦算法本身的数值行为；更高层的 KKT 推导和 selftest 验证见 [baselines.ipynb](file:///./baselines.ipynb)。

## 函数一览

| 函数 | 角色 | 复杂度 |
|---|---|---|
| `uniform_power(g, avg_power, p_max)` | 下界基线 | $O(T)$ |
| `waterfilling(g, avg_power, p_max, N)` | **主入口**，调闭式二分 | $O(T\log(1/\epsilon))$ |
| `waterfilling_closed_form(...)` | 闭式二分实现 | $O(T\cdot 200)$ |
| `waterfilling_cvxpy(...)` | cvxpy 交叉验证 | 外部求解器 |
| `total_rate(p, g, B, N, dt)` | 速率计算 | $O(T)$ |

## `uniform_power` —— 下界

```python
def uniform_power(g, avg_power, p_max):
    return np.full(T, min(avg_power, p_max), dtype=np.float64)
```

- **不看 $g$**：每时隙恒定功率 $p_t \equiv \min(P_{\text{avg}}, p_{\max})$
- 能量精确用满：$\sum p_t \Delta t = \min(P_{\text{avg}}, p_{\max})\cdot T\Delta t \le E$
- 当 $P_{\text{avg}} \le p_{\max}$（本项目 $0.05 \le 0.2$ ✓）时退化为 $p_t \equiv P_{\text{avg}}$
- 输出 float64：和注水解保持同一 dtype，避免后续比较时的类型提升陷阱

## `waterfilling_closed_form` —— 闭式二分（主求解器）

### 解的形式

$$
p_t^{*} = \mathrm{clip}\!\left(\mu - \frac{N}{g_t},\ 0,\ p_{\max}\right),\qquad \mu \text{ 由 } \sum_t p_t^{*} = P_{\text{avg}}\, T \text{ 唯一确定}
$$

### 为什么 $\sum p_t^*$ 对 $\mu$ 单调递减

$\partial p_t^* / \partial \mu \in \{0, 1\}$（clip 的中间段斜率为 1，两端为 0），所以 $\sum p_t^*$ 对 $\mu$ **单调不增**——满足二分求根条件。

### 二分区间的选择

```python
lo, hi = 1e-20, 1.0 / noise_power   # μ ∈ [1e-20, 1/N]
```

- **下界 $\mu_{\min} = 10^{-20}$**：极小水位，几乎不注水，$\sum p_t^* \approx 0 < \text{target}$
- **上界 $\mu_{\max} = 1/N$**：水位高到所有时隙都贴 $p_{\max}$（验证：$p_t^* = \min(p_{\max}, 1/\mu - N/g_t)$，当 $\mu = 1/N$ 时 $1/\mu - N/g_t = N - N/g_t$，对 $g_t \ge 1$ 的时隙 $\ge N \gg p_{\max}$，贴顶）—— 此时 $\sum p_t^* = T\cdot p_{\max} \ge \text{target}$

所以根必然落在 $[\mu_{\min}, \mu_{\max}]$ 内，二分 200 次精度达 $(1/N - 10^{-20})/2^{200} \approx 10^{-65}$，远超机器精度。

### 二分逻辑

```python
if p.sum() > target:  lo = mid   # 功率超了 → 提高水位 μ（减少注水）
else:                 hi = mid   # 功率不够 → 降水位 μ（多注水）
```

注意"超了就提高 $\mu$"是反直觉的——因为 $\sum p^*$ 对 $\mu$ 递减，功率超了说明 $\mu$ 太低，要往高调。

### `np.maximum(g, 1e-30)` 的防零

$g_t$ 可能极小（深衰落），$N/g_t$ 会爆炸成 inf 破坏二分。`np.maximum(g, 1e-30)` 给一个下地板，让 $N/g_t \le N/10^{-30} = 10^{22}$，仍在 float64 范围内。物理含义：信道再差也按 $g = 10^{-30}$ 处理（$-3000$ dB），等价于"完全不传"。

### 输出用 `hi` 而非 `mid`

```python
p = np.clip(1.0/hi - noise_power/np.maximum(g, 1e-30), 0.0, p_max)
```

二分结束时 `hi` 保证 $\sum p \le \text{target}$（不超额），而 `mid` 可能超额。用 `hi` 保证**硬能量预算不违反**——这是 selftest 检查 2（预算容差 $10^{-10}$）能通过的根因。

## `waterfilling_cvxpy` —— 独立交叉验证

### 归一化技巧（关键）

```python
g0 = np.max(g)
g_s = g / g0
n_s = noise_power / g0
```

把 $g$ 和 $N$ 同除 $g_{\max}$：

$$
\frac{g_t\, p_t}{N} = \frac{(g_t/g_0)\, p_t}{N/g_0}
$$

SNR 不变 → 速率不变，但 $g_s$ 的动态范围从 $10^7$ 压到 $1$，$\log$ 目标条件数大幅改善。CLARABEL 在归一化后有时能解，SCS 更鲁棒。

### 求解器回退链

```python
for solver in [cp.CLARABEL, cp.SCS]:    # 先试 CLARABEL（快、精确）
    try: prob.solve(solver=solver)
    except: continue
if 都失败: return waterfilling_closed_form(g, ...)   # 兜底
```

**实测**：$g$ 跨 7 个数量级时，CLARABEL 在未归一化下直接 SolverError；归一化后能解；SCS 对 $\log$ 目标更鲁棒。两者都失败的概率极低，仍兜底回退闭式——保证这个函数**永不返回 None**。

### 与闭式解的对拍

selftest 检查 4：闭式解 vs cvxpy 解，速率相对差 $< 10^{-4}$。实测典型偏差 $\sim 6\times10^{-7}$（源自 float64 下残余的求和误差，不是算法差异）。

## `total_rate` —— 速率计算

$$
R = \sum_{t=1}^{T} B\,\log_2\!\left(1 + \frac{p_t\, g_t}{N}\right)\Delta t \quad [\text{bits}]
$$

- 输入 $p$, $g$ 是同形状数组，逐元素算 SNR 再 $\log_2$，最后求和乘 $B\Delta t$
- **不检查约束**：纯评估函数，给定 $p$ 算速率，不管 $p$ 是否满足预算/边界
- 输出 float（标量），单位 bits；下游 `/1e6` 转 Mb 打印

### 数值注意

`np.log2(1 + p*g/N)` 当 $p\,g/N \ll 1$（低 SNR）时趋近 $p\,g/(N\ln 2)$，数值稳定；当 $p\,g/N \gg 1$（高 SNR）时 $\log_2$ 本身无问题。整体无数值风险，**唯一陷阱是 float32 求和误差**（见下）。

## float32 陷阱（已修复）

### 现象

轨迹 npz 存的是 float32。早期 `waterfilling_closed_form` 未做 dtype 转换时：

```python
g = np.load(...)['g'][0]          # float32
p = np.clip(1/mid - N/g, 0, pmax) # float32（NumPy 2.x 弱标量提升）
if p.sum() > target:              # float32 的 sum 带 ~1e-6 误差
```

二分的停止判据被 $10^{-6}$ 量级的求和误差污染，导致最终解的 $\sum p$ 微微超过 target 约 $10^{-7}$ 相对——物理可忽略（速率不可见），但违反硬约束的形式正确性。

### 根因

NumPy 2.x 的"弱标量提升"（NEP 50 之前的兼容行为）让 `1/mid - N/g` 在 float32 输入下全流程 float32，`p.sum()` 用 float32 累加 1000 个元素，累计舍入误差 $\sim \sqrt{1000}\times 2^{-23} \approx 10^{-6}$。

### 修复

两个注水入口都加一行：

```python
g = np.asarray(g, dtype=np.float64)
```

后续 `1/mid`、`N/g`、`p.sum()` 全在 float64 下做，求和误差降到 $10^{-15}$ 量级。selftest 的预算容差 $10^{-10}$ 才能通过。

### 为什么不在存 npz 时就存 float64

- 体积翻倍（13 MB → 26 MB），压缩后差异更小但仍浪费
- float32 的 ~7 位有效数字对 $g_t$（功率增益，跨 7 个数量级）**存储**够用
- 数值精度问题只在**求和/二分停止判据**这一步出现，下游侧转 float64 是局部、便宜的防御

In [1]:
# 演示：逐函数调用，看每个函数的输出
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from config import SceneConfig
from channel.trace_io import load_traces
from baselines.power_alloc import (
    uniform_power, waterfilling, waterfilling_closed_form,
    waterfilling_cvxpy, total_rate
)

cfg = SceneConfig()
g_test, _, _ = load_traces('traces/sionna_speed120_test.npz')
g = g_test[0]                       # 取第一条轨迹
print(f'g: shape={g.shape}, dtype={g.dtype}, range=[{g.min():.2e}, {g.max():.2e}]')

# 1. 均匀
p_u = uniform_power(g, cfg.avg_power, cfg.p_max)
print(f'\nuniform: p_t={p_u[0]:.4f} (恒定), Σp·dt={p_u.sum()*cfg.dt_s:.6e} (E={cfg.avg_power*cfg.T*cfg.dt_s:.6e})')

# 2. 注水（闭式，主入口）
p_w = waterfilling(g, cfg.avg_power, cfg.p_max, cfg.noise_power)
print(f'waterfilling (closed-form): Σp·dt={p_w.sum()*cfg.dt_s:.6e}, p_max={p_w.max():.4f}, p_min={p_w.min():.4f}')

# 3. cvxpy 交叉验证
p_c = waterfilling_cvxpy(g, cfg.avg_power, cfg.p_max, cfg.noise_power)
r_w = total_rate(p_w, g, cfg.bandwidth_hz, cfg.noise_power, cfg.dt_s)
r_c = total_rate(p_c, g, cfg.bandwidth_hz, cfg.noise_power, cfg.dt_s)
print(f'waterfilling (cvxpy):      Σp·dt={p_c.sum()*cfg.dt_s:.6e}')
print(f'\n速率对比: 闭式={r_w/1e6:.6f} Mb, cvxpy={r_c/1e6:.6f} Mb, 相对差={abs(r_w-r_c)/r_c:.2e}')

# 4. 速率
r_u = total_rate(p_u, g, cfg.bandwidth_hz, cfg.noise_power, cfg.dt_s)
print(f'\n速率: uniform={r_u/1e6:.4f} Mb, waterfilling={r_w/1e6:.4f} Mb, 增益=+{(r_w/r_u-1)*100:.2f}%')


g: shape=(1000,), dtype=float32, range=[3.52e-12, 1.06e-06]

uniform: p_t=0.0500 (恒定), Σp·dt=5.000000e-02 (E=5.000000e-02)
waterfilling (closed-form): Σp·dt=5.000000e-02, p_max=0.1232, p_min=0.0000
waterfilling (cvxpy):      Σp·dt=5.000000e-02

速率对比: 闭式=9.630149 Mb, cvxpy=9.630149 Mb, 相对差=1.61e-08

速率: uniform=8.0109 Mb, waterfilling=9.6301 Mb, 增益=+20.21%
